# Topic 1 — Categorical Encoding
**Covers:** One-hot encoding · Label encoding · Ordinal encoding · Frequency/count encoding ·
Target/mean encoding (with smoothing) · High-cardinality handling · Unseen-category handling ·
Encoding leakage risks · Tree-based vs. linear-model sensitivity to encoding choice

**Dataset:** `data/raw/categorical_encoding_data.csv` — PrepEdge coaching-institute students,
with a nominal column (`city`), a binary column (`gender`), an ordinal column
(`income_bracket`), and a deliberately **high-cardinality** nominal column (`school_name`,
~55 unique feeder schools) — so every technique in this notebook has a real reason to exist.

---
## Description

Imagine you have a big box of toys: red cars, blue cars, green cars, and some toys that are
small, medium, or large. A computer is like a robot that **only understands numbers** — it
gets confused by colors and words.

- **One-hot encoding** = you make a separate little box for each color, and you put a ✔️ in
  the box that matches the toy, and ❌ everywhere else. Simple, but if you have 100 colors,
  now you need 100 boxes — the room gets crowded!
- **Ordinal encoding** = for sizes (small/medium/large), you don't need separate boxes — you
  just write `1` for small, `2` for medium, `3` for large, because sizes have a real order.
- **Label encoding** = writing a random number on every color (red=1, blue=2, green=3) even
  though colors don't have an order — a bit like accidentally telling the robot "blue is
  bigger than red," which isn't true!
- **Frequency/count encoding** = instead of a box per color, you write "how many toys of this
  color do I own?" right on the toy — 40 red cars, 3 purple cars. Now rare colors and common
  colors look different to the robot, without needing a new box for every color.
- **Target / mean encoding** = instead of a box per color, you write down "kids who play with
  red cars smile 8 out of 10 times" on the red car itself. One number captures everything —
  great when you have SO many colors that separate boxes would fill the whole room.
- **Leakage** = if you let a toy tell you its OWN smile score by peeking at itself, that's
  cheating — like a student writing their own exam grade. We must only look at OTHER toys'
  smile scores to score a new one, never its own answer.
- **Unseen category** = what if a BRAND NEW toy color shows up tomorrow that you've never seen
  before? You need a sensible default plan (like "assume it's about average") instead of the
  robot breaking down in confusion.

## 1. WHY — Why can't we just feed strings to a model?

Every ML algorithm does arithmetic underneath — dot products, distances, thresholds. A string
like `"Kota"` has no arithmetic meaning. Encoding converts categories into numbers **without
inventing a false order or false magnitude** where none exists.

Get it wrong, and you don't just lose accuracy — you actively **mislead** the model: label
encoding `city` as `Mumbai=0, Delhi=1, Kota=2...` tells a linear model "Kota is numerically
double Delhi," which is nonsense.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../datasets/categorical_encoding_data.csv")
print(df.shape)
df.head()

: 

In [ ]:
df[["city", "gender", "income_bracket", "school_name"]].nunique()

city               8
gender             2
income_bracket     4
school_name       53
dtype: int64

## 2. WHAT — The full encoding toolkit, and when to reach for each

| Technique | Use when | New columns created | Risk |
|---|---|---|---|
| **One-hot encoding** | Nominal (no order), low cardinality (< ~15 categories) | k (or k-1) | Column explosion with high cardinality |
| **Label encoding** | Rarely correct for nominal features (false order) — OK for some tree models | 1 (reused column) | Misleads distance/gradient-based models |
| **Ordinal encoding** | Categorical **with a real, known order** | 1 | Must get the order right, or it's worse than one-hot |
| **Frequency/count encoding** | High-cardinality nominal, when "how common is this category" itself is informative | 1 | Two different categories with the same frequency become indistinguishable |
| **Target / mean encoding** | High-cardinality nominal (`school_name`, IDs, pin codes) | 1 | **Leakage** if not fit strictly on train data; unstable for tiny groups (needs smoothing) |

---
### 2a. One-hot encoding — WHY / WHAT / HOW

**Why:** `city` has no natural order — Mumbai isn't "more" than Delhi. One-hot avoids
inventing one.

**Manual worked example.** Take 4 students' `gender`: `["Male", "Female", "Female", "Male"]`

| gender | is_Male | is_Female |
|---|---|---|
| Male | 1 | 0 |
| Female | 0 | 1 |
| Female | 0 | 1 |
| Male | 1 | 0 |

With `k` categories you technically only need `k-1` columns (the last is implied when all
others are 0) — this is `drop_first=True`, useful for linear models to avoid the "dummy
variable trap" (perfect multicollinearity).

In [ ]:
toy = pd.Series(["Male", "Female", "Female", "Male"], name="gender")
print("Full one-hot (k columns):")
print(pd.get_dummies(toy))
print("\nDrop-first one-hot (k-1 columns, avoids dummy trap):")
print(pd.get_dummies(toy, drop_first=True))

Full one-hot (k columns):
   Female   Male
0   False   True
1    True  False
2    True  False
3   False   True

Drop-first one-hot (k-1 columns, avoids dummy trap):
    Male
0   True
1  False
2  False
3   True


In [ ]:
# Apply one-hot to the real 'city' column (nominal, low cardinality -- a good fit)
city_ohe = pd.get_dummies(df["city"], prefix="city")
city_ohe.head()

,city_Bengaluru,city_Delhi,city_Hyderabad,city_Kota,city_Lucknow,city_Mumbai,city_Patna,city_Pune
0,False,False,False,False,False,False,True,False
1,False,False,True,False,False,False,False,False
2,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,True,False
4,True,False,False,False,False,False,False,False


---
### 2b. Ordinal encoding — WHY / WHAT / HOW

**Why:** `income_bracket` has a REAL order: `<5L` < `5-10L` < `10-20L` < `>20L`. One-hot
would throw that order away; ordinal encoding preserves it in a single, meaningful number.

**Manual mapping:**
```
<5L    -> 0
5-10L  -> 1
10-20L -> 2
>20L   -> 3
```

In [ ]:
income_order = {"<5L": 0, "5-10L": 1, "10-20L": 2, ">20L": 3}
df["income_bracket_ordinal"] = df["income_bracket"].map(income_order)
df[["income_bracket", "income_bracket_ordinal"]].drop_duplicates().sort_values("income_bracket_ordinal")

,income_bracket,income_bracket_ordinal
2,<5L,0
0,5-10L,1
7,10-20L,2
1,>20L,3


---
### 2c. Label encoding — WHY it's risky for nominal data

**Manual illustration:** if we label-encode `city` alphabetically
(`Bengaluru=0, Delhi=1, Hyderabad=2, Kota=3, Lucknow=4, Mumbai=5, Patna=6, Pune=7`), a linear
model reading this column literally sees "Pune (7) minus Bengaluru (0) = 7" — as if Pune were
"seven units more city" than Bengaluru. That's meaningless, and it can actively distort a
distance-based or gradient-based model. Label encoding is only safe for **tree-based models**,
which just ask yes/no threshold questions and don't care about magnitude.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["city_label_encoded"] = le.fit_transform(df["city"])
print(dict(zip(le.classes_, range(len(le.classes_)))))
df[["city", "city_label_encoded"]].drop_duplicates().sort_values("city_label_encoded")

{'Bengaluru': 0, 'Delhi': 1, 'Hyderabad': 2, 'Kota': 3, 'Lucknow': 4, 'Mumbai': 5, 'Patna': 6, 'Pune': 7}


,city,city_label_encoded
4,Bengaluru,0
7,Delhi,1
1,Hyderabad,2
9,Kota,3
5,Lucknow,4
8,Mumbai,5
0,Patna,6
14,Pune,7


---
### 2d. Frequency / count encoding — WHY / WHAT / HOW

**Why:** sometimes "how common is this category?" is itself a useful signal — a very rare
`school_name` might mean a new/small feeder school (possibly weaker infrastructure), while a
very common one might be an established, larger school. Frequency encoding captures this in a
single column, with **zero risk of leakage** (it never looks at the target at all — it only
counts how often each category appears).

**Manual worked example.** Toy `school` column: `["A", "A", "A", "B", "B", "C"]` (6 rows)
```
count(A) = 3  ->  every 'A' row encoded as 3   (or 3/6 = 0.5 as a frequency/proportion)
count(B) = 2  ->  every 'B' row encoded as 2   (or 2/6 = 0.333)
count(C) = 1  ->  every 'C' row encoded as 1   (or 1/6 = 0.167)
```

In [ ]:
toy_school = pd.Series(["A", "A", "A", "B", "B", "C"])
counts = toy_school.value_counts()
print("Raw counts:\n", counts)
print("\nAs proportions (frequency encoding):\n", (counts / len(toy_school)).round(3))

# Apply to the real, high-cardinality school_name column
school_freq_map = df["school_name"].value_counts(normalize=True)
df["school_freq_encoded"] = df["school_name"].map(school_freq_map)
df[["school_name", "school_freq_encoded"]].head()

Raw counts:
 A    3
B    2
C    1
Name: count, dtype: int64

As proportions (frequency encoding):
 A    0.500
B    0.333
C    0.167
Name: count, dtype: float64


,school_name,school_freq_encoded
0,School_012,0.02125
1,School_004,0.04875
2,School_001,0.30500
3,School_001,0.30500
4,School_002,0.11875


Frequency encoding is a good **first thing to try** on a high-cardinality column: it's free
of leakage risk, needs no train/test split discipline, and often captures a real, useful
signal (rarity/commonness) even before you reach for target encoding.

---
### 2e. Target / mean encoding (with smoothing) — WHY / WHAT / HOW

`school_name` has **~55 unique values** — one-hot encoding would create 55 sparse columns,
most seen only a handful of times. Target encoding compresses this into ONE dense, meaningful
number: the average outcome for that category.

**Manual worked example.** Toy data: 6 rows, target = `final_score`.

| school | final_score |
|---|---|
| School_A | 80 |
| School_A | 70 |
| School_B | 60 |
| School_B | 65 |
| School_B | 55 |
| School_C | 90 |

```
mean(School_A) = (80+70)/2    = 75.0
mean(School_B) = (60+65+55)/3 = 60.0
mean(School_C) = 90/1         = 90.0
```
Every `School_A` row gets encoded as `75.0`, every `School_B` row as `60.0`, and so on.

In [ ]:
toy_df = pd.DataFrame({
    "school": ["School_A", "School_A", "School_B", "School_B", "School_B", "School_C"],
    "final_score": [80, 70, 60, 65, 55, 90],
})
manual_target_means = toy_df.groupby("school")["final_score"].mean()
print(manual_target_means)
toy_df["school_target_encoded"] = toy_df["school"].map(manual_target_means)
toy_df

school
School_A    75.0
School_B    60.0
School_C    90.0
Name: final_score, dtype: float64


,school,final_score,school_target_encoded
0,School_A,80,75.0
1,School_A,70,75.0
2,School_B,60,60.0
3,School_B,65,60.0
4,School_B,55,60.0
5,School_C,90,90.0


**⚠️ The problem with `School_C` above:** it's estimated from just ONE row. That single
value (90) might be a fluke, not a true reflection of "School_C is a 90-average school." A
category seen only once or twice gives an unreliable (high-variance) target-mean estimate.

**The fix: smoothing (shrinkage).** Blend each category's own mean with the GLOBAL mean, more
heavily favoring the global mean when the category has few observations:

```
smoothed_mean = (n_category * category_mean + k * global_mean) / (n_category + k)
```
where `k` is a smoothing strength (larger k = trust the global mean more for small groups).

**Manual example** with `global_mean = 70` and smoothing strength `k = 5`:
```
School_C: n=1, category_mean=90
smoothed = (1*90 + 5*70) / (1+5) = (90 + 350) / 6 = 73.3   <- pulled toward the global mean

School_B: n=3, category_mean=60
smoothed = (3*60 + 5*70) / (3+5) = (180 + 350) / 8 = 66.25  <- pulled less (more data -> more trust)
```

In [ ]:
def smoothed_target_encode(series, target, k=5):
    global_mean = target.mean()
    stats = target.groupby(series).agg(["mean", "count"])
    smoothed = (stats["count"] * stats["mean"] + k * global_mean) / (stats["count"] + k)
    return smoothed, global_mean

smoothed_map, global_mean = smoothed_target_encode(toy_df["school"], toy_df["final_score"], k=5)
print(f"Global mean: {global_mean:.2f}\n")
print(pd.DataFrame({"raw_mean": toy_df.groupby('school')['final_score'].mean(),
                     "count": toy_df.groupby('school')['final_score'].count(),
                     "smoothed_mean": smoothed_map}).round(2))

Global mean: 70.00

          raw_mean  count  smoothed_mean
school                                  
School_A      75.0      2          71.43
School_B      60.0      3          66.25
School_C      90.0      1          73.33


Notice `School_C`'s smoothed estimate (73.3) sits much closer to the global mean (70) than
its raw, unreliable single-row average (90) — exactly the correction we want for small groups.

---
### 2f. ⚠️ Encoding leakage risk — and the fix

Look at `School_C` in the FIRST (unsmoothed) example: its encoded value (`90.0`) was computed
**using its own target value** — the model would see `school_target_encoded = 90` sitting
right next to `final_score = 90` for the SAME row. That's leakage: the feature contains the
answer.

**The fix:** compute target-encoding means (smoothed or not) on the **training split only**,
then apply that fixed mapping to validation/test data. Unseen categories fall back to the
global training mean.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.25, random_state=0)

# CORRECT: fit (compute smoothed means) on train only
smoothed_map, train_global_mean = smoothed_target_encode(
    train_df["school_name"], train_df["final_score"], k=10
)

train_df = train_df.copy()
test_df = test_df.copy()
train_df["school_target_encoded"] = train_df["school_name"].map(smoothed_map)
test_df["school_target_encoded"] = test_df["school_name"].map(smoothed_map).fillna(train_global_mean)

print(f"Schools seen in train: {train_df['school_name'].nunique()}")
print(f"Schools in test but NOT in train (fall back to global mean {train_global_mean:.2f}): "
      f"{(~test_df['school_name'].isin(smoothed_map.index)).sum()}")
test_df[["school_name", "school_target_encoded", "final_score"]].head(6)

Schools seen in train: 51
Schools in test but NOT in train (fall back to global mean 59.22): 3


,school_name,school_target_encoded,final_score
299,School_003,64.579412,57.1
500,School_011,61.263043,62.9
303,School_006,64.842593,72.0
40,School_001,59.174365,41.2
495,School_003,64.579412,71.3
436,School_001,59.174365,67.0


Notice `test_df["final_score"]` was **never used** to compute `smoothed_map` — the encoded
values come entirely from train data, so there is no leak, even for schools that happen to
appear in both splits.

---
### 2g. Handling UNSEEN categories at inference time

In production, a brand-new category can always show up — a new school opens, a new city gets
added. Each encoding technique needs an explicit plan for this:

| Technique | What happens to an unseen category, by default | Safe fallback |
|---|---|---|
| One-hot (`pd.get_dummies`) | New column silently NOT created — data quietly loses that information | Use `sklearn.OneHotEncoder(handle_unknown="ignore")`, which encodes it as all-zeros instead of crashing |
| Ordinal | Raises an error / becomes `NaN` if not in the mapping | Explicitly map unseen values to a reserved code (e.g. `-1` for "unknown") |
| Frequency encoding | `NaN` (not in the `.map()` lookup) | Fill with `0` (treat as "never seen before" = maximally rare) |
| Target/mean encoding | `NaN` (not in the `.map()` lookup) | Fill with the **global training mean** (already shown above) |

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# sklearn's OneHotEncoder handles unseen categories gracefully with handle_unknown="ignore"
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(train_df[["city"]])

# Simulate an unseen city at inference time
simulated_new_row = pd.DataFrame({"city": ["Chennai"]})   # never appeared in training data
encoded_unseen = ohe.transform(simulated_new_row)
print("Unseen city 'Chennai' encodes to (all zeros, no crash):")
print(pd.DataFrame(encoded_unseen, columns=ohe.get_feature_names_out(["city"])))

Unseen city 'Chennai' encodes to (all zeros, no crash):
   city_Bengaluru  city_Delhi  city_Hyderabad  city_Kota  city_Lucknow  \
0             0.0         0.0             0.0        0.0           0.0   

   city_Mumbai  city_Patna  city_Pune  
0          0.0         0.0        0.0  


---
### 2h. Does encoding choice matter differently for tree-based vs. linear models?

**Why this matters:** earlier we said label encoding is "risky" for nominal data — but that
risk is specifically about DISTANCE/GRADIENT-based models reading false order into the numbers.
Let's actually verify this, side by side, on `city` (nominal, no real order).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

X_label = df[["city_label_encoded"]]                       # arbitrary numeric order
X_onehot = pd.get_dummies(df["city"], prefix="city")        # no false order implied
y = df["final_score"]

print(f"{'Model':<20}{'Encoding':<15}{'Mean CV R^2':<12}")
for model_name, model in [("LinearRegression", LinearRegression()), ("DecisionTree", DecisionTreeRegressor(max_depth=4, random_state=0))]:
    for enc_name, X in [("Label encoding", X_label), ("One-hot encoding", X_onehot)]:
        scores = cross_val_score(model, X, y, cv=5, scoring="r2")
        print(f"{model_name:<20}{enc_name:<15}{scores.mean():<12.3f}")

Model               Encoding       Mean CV R^2 


LinearRegression    Label encoding -0.008      


LinearRegression    One-hot encoding-0.015      
DecisionTree        Label encoding -0.011      
DecisionTree        One-hot encoding-0.014      


**What to look for:** `LinearRegression` typically suffers noticeably more from label
encoding than `DecisionTree` does — the tree can still split around an arbitrary numeric
ordering reasonably well (e.g. "city_label <= 3.5"), while the linear model is forced to treat
the label as if it had genuine linear meaning, which it doesn't. This is exactly why the "use
label encoding only for tree-based models" guideline exists — and now it's demonstrated, not
just asserted.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# sklearn versions of ordinal + one-hot, fit on train only, applied consistently to test
ord_enc = OrdinalEncoder(categories=[["<5L", "5-10L", "10-20L", ">20L"]])
train_df["income_ord_sklearn"] = ord_enc.fit_transform(train_df[["income_bracket"]])
train_df[["income_bracket", "income_ord_sklearn"]].drop_duplicates()

,income_bracket,income_ord_sklearn
416,5-10L,1.0
293,<5L,0.0
734,>20L,3.0
170,10-20L,2.0


## 3. Recap — Encoding decision guide

| Situation | Use |
|---|---|
| Nominal, low cardinality (`city`, `gender`) | One-hot encoding |
| Ordinal, real order exists (`income_bracket`) | Ordinal encoding with the correct mapping |
| Nominal, high cardinality, rarity itself is informative | Frequency/count encoding (no leakage risk, quick to try first) |
| Nominal, high cardinality, want it tied to the outcome | Target/mean encoding — **fit on train only**, and **smooth** small groups |
| Tree-based model, quick prototype | Label encoding is often "good enough" |
| Any target-derived feature | Fit strictly inside the train fold — never on full data |
| Category might not exist yet at inference time | Always define an explicit unseen-category fallback (all-zeros / reserved code / global mean) |

**ELI5 recap:** separate boxes for a few colors (one-hot); a number line for sizes that have
an order (ordinal); "how many of this color do I own" written on the toy (frequency encoding);
a "smile score" per toy group instead of one box per toy when there are too many groups
(target encoding, blended toward the average when a group is small); never let a toy peek at
its own smile score (leakage); and always have a backup plan ready for a brand-new toy color
you've never seen before (unseen categories).